In [1]:
import os
import torch

gpu_list = [5]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from torch.nn import Linear
from model.layers import Add, Clone
from model.ViT import Attention
import torch.nn.functional as F
from einops import rearrange
import torch.nn as nn
import torchvision.models as models
from torch_geometric.nn import GATv2Conv, LayerNorm
from model.ViT import Mlp, VisionTransformer


class GraphTransformer(nn.Module):
    def __init__(self, cell_dim=80, vit_depth=3, proto=False, ensemble=False):
        super(GraphTransformer, self).__init__()
        self.resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.resnet18 = torch.nn.Sequential(*list(self.resnet18.children())[:-1])
        
        self.embed_dim = 32 * 8
        self.head = 8
        self.dropout = 0.3
        
        self.conv1 = GATv2Conv(in_channels=512, out_channels=int(self.embed_dim/self.head), heads=self.head)
        self.norm1 = LayerNorm(in_channels=self.embed_dim)
        
        self.cell_transformer = VisionTransformer(num_classes=cell_dim, embed_dim=self.embed_dim, depth=vit_depth,
                                                  mlp_head=True, drop_rate=self.dropout, attn_drop_rate=self.dropout)
        self.proto = proto
        if self.proto:
            self.cell_proto = nn.Parameter(torch.zeros(1, 150, self.embed_dim))
            self.cell_qkv = Linear(self.embed_dim, self.embed_dim*2)
            self.cell_att = Attention(dim=self.embed_dim, num_heads=self.head, attn_drop=self.dropout, proj_drop=self.dropout)
            self.add2 = Add()
            self.clone2 = Clone()
            self.task_norm_3 = LayerNorm(self.embed_dim, eps=1e-6)
            self.task_norm_4 = LayerNorm(self.embed_dim, eps=1e-6)
            self.cell_att_W = Linear(self.embed_dim, self.embed_dim)
            torch.nn.init.xavier_uniform_(self.cell_proto)
            
        self.ensemble = ensemble
        if self.ensemble:
            self.spot_fc = Linear(in_features=512, out_features=256)
            self.spot_head = Mlp(in_features=256, hidden_features=512*2, out_features=cell_dim)
            self.local_head = Mlp(in_features=256, hidden_features=512*2, out_features=cell_dim)
            self.fused_head = Mlp(in_features=256, hidden_features=512*2, out_features=cell_dim)
        
    def forward(self, x, edge_index):
        x_spot = self.resnet18(x)
        x_spot = x_spot.squeeze()
        
        x_local = self.conv1(x=x_spot, edge_index=edge_index)
        x_local = self.norm1(x_local)
        
        x_local = x_local.unsqueeze(0)
        
        if self.proto:
            x_cell1, x_cell2 = self.clone2(x_local, 2)
            x_cell_qkv = self.cell_qkv(self.cell_proto)
            x_cell_k, x_cell_v = rearrange(x_cell_qkv, 'b n (qkv h d) -> qkv b h n d', qkv=2, h=8)
            x_cell = self.add2([x_cell1, self.cell_att_W(self.cell_att(x=x_cell2, out_k=x_cell_k, out_v=x_cell_v))])
            x_cell = self.task_norm_4(x_cell)
            x_cell = self.task_norm_3(x_cell + F.relu(x_cell))
        else:
            x_cell = x_local
        
        if self.ensemble:
            x_spot = self.spot_fc(x_spot)
            cell_predication_spot = self.spot_head(x_spot)
            x_local = x_local.squeeze(0)
            cell_prediction_local = self.local_head(x_local)
            cell_prediction_global, x_global = self.cell_transformer(x_cell)
            cell_prediction_global = cell_prediction_global.squeeze()
            x_global = x_global.squeeze()
            cell_prediction_fused = self.fused_head((x_spot+x_local+x_global)/3.0)
            cell_prediction = (cell_predication_spot + cell_prediction_local + cell_prediction_global + cell_prediction_fused) / 4.0
        else:  
            cell_prediction, _ = self.cell_transformer(x_cell)
            cell_prediction = cell_prediction.squeeze()
            
        cell_prediction = torch.relu(cell_prediction)
        
        return cell_prediction

In [8]:
from glob import glob
tif_list = glob('/data1/r20user3/shared_project/Hist2Cell/code/training/train_test_splits/her2st/test*')
tif_list.sort()
test_slides = list()
for tif in tif_list:
    tif_path = tif.split('_')[-1].split('.')[0]
    test_slides.append(tif_path)
test_slides

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

In [9]:
from torch_geometric.data import Batch
from torch_geometric.loader import NeighborLoader
import torch_geometric
torch_geometric.typing.WITH_PYG_LIB = False
from tqdm import tqdm
import numpy as np
from scipy.stats import pearsonr
import joblib
    
for case in test_slides:
    print(case)
    ensemble = True
    vit_depth = 3
    celltype_num = 39
    model = GraphTransformer(vit_depth=vit_depth, ensemble=ensemble, cell_dim=celltype_num)
    checkpoint = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/training/model_weights/her2st_epoch100_lr1e-4_2hop_ensemble_onlycell_"+case+"_best_cell_all_abundance_average.pth")
    model.load_state_dict(checkpoint)
    model = model.to(device)
    
    test_set = "/data1/r20user3/shared_project/Hist2Cell/code/training/train_test_splits/her2st/test_leave_"+case+".txt"
    data_path = "/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/her2st"
    hop = 2
    subgraph_bs = 16   
    
    criterion = nn.MSELoss().to(device)
    
    test_slides = open(test_set).read().split('\n')

    test_graph_list = list()
    for item in test_slides:
        test_graph_list.append(torch.load(os.path.join(data_path, item+'.pt')))
    test_dataset = Batch.from_data_list(test_graph_list)
    
    print(test_slides)
    # continue    

    test_loader = NeighborLoader(
        test_dataset,
        num_neighbors=[-1]*hop,
        batch_size=subgraph_bs,
        directed=False,
        input_nodes=None,
        shuffle=False,
        # num_workers=8,
        # pin_memory=True, 
        # prefetch_factor=2,
    )

    with torch.no_grad():
        model.eval()

        test_sample_num = 0
        test_cell_pred_array = []
        test_cell_label_array = []
        test_cell_pos_array = []
        test_loss, test_cell_abundance_loss = 0, 0
        
        for graph in tqdm(test_loader):
            x = graph.x.to(device)
            y = graph.y.to(device)
            pos = graph.pos.to(device)
            edge_index = graph.edge_index.to(device)
            cell_label = y[:, 250:]
            cell_pred = model(x=x, edge_index=edge_index)
            cell_loss = criterion(cell_pred, cell_label)

            loss = cell_loss
                
            center_num = len(graph.input_id)
            center_cell_label = cell_label[:center_num, :]
            center_cell_pred = cell_pred[:center_num, :]
            center_cell_pos = pos[:center_num, :]
            
            test_cell_label_array.append(center_cell_label.squeeze().cpu().detach().numpy())
            test_cell_pred_array.append(center_cell_pred.squeeze().cpu().detach().numpy())
            test_cell_pos_array.append(center_cell_pos.squeeze().cpu().detach().numpy())
            test_sample_num = test_sample_num + center_num
            
            test_loss += loss.item() * center_num
            test_cell_abundance_loss += cell_loss.item() * center_num

        test_cell_abundance_loss = test_cell_abundance_loss / test_sample_num
    
        if len(test_cell_pred_array[-1].shape) == 1:
            test_cell_pred_array[-1] = np.expand_dims(test_cell_pred_array[-1], axis=0)
        test_cell_pred_array = np.concatenate(test_cell_pred_array)
        if len(test_cell_label_array[-1].shape) == 1:
            test_cell_label_array[-1] = np.expand_dims(test_cell_label_array[-1], axis=0)
        test_cell_label_array = np.concatenate(test_cell_label_array)
        if len(test_cell_pos_array[-1].shape) == 1:
            test_cell_pos_array[-1] = np.expand_dims(test_cell_pos_array[-1], axis=0)
        test_cell_pos_array = np.concatenate(test_cell_pos_array)
    
        test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = 0.0
        test_cell_abundance_pos_pearson_count = 0   
        test_cell_pearson_list = []
        for i in range(celltype_num):
            r, p = pearsonr(test_cell_pred_array[:, i], test_cell_label_array[:, i])
            if r > 0.0 and p <= 0.05:
                test_cell_abundance_pos_pearson_count = test_cell_abundance_pos_pearson_count + 1
                test_cell_abundance_pos_pearson_average = test_cell_abundance_pos_pearson_average + r
            if np.isnan(r):
                r = 0.0
            test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average + r
            test_cell_pearson_list.append(r)
        if test_cell_abundance_pos_pearson_count >= 1:
            test_cell_abundance_pos_pearson_average /= test_cell_abundance_pos_pearson_count
        else:
            test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average / celltype_num

        print(case)
        print("saving " + "best cell all abundance average " + str(test_cell_abundance_all_pearson_average))
        
    
    Predictions = dict()

    for slide_no in range(len(test_slides)):
        indices = np.where(test_dataset.batch.numpy() == slide_no)
        test_cell_pred_array_sub = test_cell_pred_array[indices]
        test_cell_label_array_sub = test_cell_label_array[indices]
        test_cell_pos_arraay_sub = test_cell_pos_array[indices]
        
        test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = 0.0
        test_cell_abundance_pos_pearson_count = 0   
        test_cell_pearson_list = []
        for i in range(celltype_num):
            r, p = pearsonr(test_cell_pred_array_sub[:, i], test_cell_label_array_sub[:, i])
            if r > 0.0 and p <= 0.05:
                test_cell_abundance_pos_pearson_count = test_cell_abundance_pos_pearson_count + 1
                test_cell_abundance_pos_pearson_average = test_cell_abundance_pos_pearson_average + r
            if np.isnan(r):
                r = 0.0
            test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average + r
            test_cell_pearson_list.append(r)
        if test_cell_abundance_pos_pearson_count >= 1:
            test_cell_abundance_pos_pearson_average /= test_cell_abundance_pos_pearson_count
        else:
            test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average / celltype_num
        
        Predictions[test_slides[slide_no]] = {
            'cell_abundance_predictions': test_cell_pred_array_sub,
            'cell_abundance_labels': test_cell_label_array_sub,
            'coords': test_cell_pos_arraay_sub,
        }
        
        print(test_slides[slide_no]) 
        print(test_cell_abundance_all_pearson_average)    
        

    # save_path = "/data1/r20user3/shared_project/Hist2Cell/code/analysis/inference/her2st/her2st_epoch100_lr1e-4_2hop_ensemble_onlycell_"+case+"_best_cell_all_abundance_average.pkl"

    # joblib.dump(Predictions, save_path)
    

A


/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:50: UserWarning: Using '{self.__class__.__name__}' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn("Using '{self.__class__.__name__}' without a "


['A1', 'A2', 'A3', 'A4', 'A5', 'A6']


100%|██████████| 130/130 [00:17<00:00,  7.58it/s]


A
saving best cell all abundance average 0.21423247901531936
A1
0.21509892323853302
A2
0.2204816642102938
A3
0.24419865778994057
A4
0.20090326525472127
A5
0.2834152828421037
A6
0.17116882839857844
B
['B1', 'B2', 'B3', 'B4', 'B5', 'B6']


...

F
saving best cell all abundance average 0.22498993478373563
F1
0.1939848219360663
F2
0.30261915037340675
F3
0.21303037370083458
G
['G1', 'G2', 'G3']


100%|██████████| 86/86 [00:11<00:00,  7.34it/s]


G
saving best cell all abundance average 0.5889143434183856
G1
0.5881994137729573
G2
0.5733271388028329
G3
0.6154202915208956
H
['H1', 'H2', 'H3']


100%|██████████| 108/108 [00:15<00:00,  6.86it/s]

H
saving best cell all abundance average 0.5809168773340425
H1
0.5762457192915742
H2
0.5827410935667897
H3
0.5853570134640024


In [5]:
test_slides

['A1', 'A2', 'A3', 'A4', 'A5', 'A6']